# Positional Prune Entropy Heatmap Visualization

This notebook visualizes attention head entropy distribution using the POSITIONAL_PRUNE processor.

## Overview
- Calibrates the POSITIONAL_PRUNE processor to collect entropy statistics
- Creates heatmaps showing entropy values across layers and heads
- Highlights which heads/positions are marked for pruning

## 1. Setup and Imports

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from omegaconf import OmegaConf
from collections import defaultdict

from segment_anything import SamPredictor, sam_model_registry
from segment_anything.modeling.image_encoder import Attention as EncoderSamAttention
from segment_anything.modeling.transformer import Attention as DecoderAttention
from train.segment_anything_training.modeling.image_encoder import Attention as EncoderAttentionTraining
from seginw.segment_anything.modeling.image_encoder import Attention as EncoderAttention

from processors import get_encoder_processor
from small_engine import Engine, override_args

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print("All imports successful!")

/home/chauht2/SAM_Quantization/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/chauht2/SAM_Quantization/.venv/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/chauht2/SAM_Quantization/.venv/lib/python3.10/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/home/chauht2/SAM_Quantization/.venv/lib/python3.10/site-packages/segment_anything/modeling/tiny_vit_sam.py:662: UserWarning: Overwri

All imports successful!


## 2. Configuration

In [2]:
# Configuration parameters
CONFIG_FILE = 'quant/config/hq44k/rtn.yaml'
PROCESSOR_NAME = 'POSITIONAL_PRUNE'
PERCENT_ENTROPY = 0.3  # Percentage of heads to prune based on entropy
NUM_CALIB_SAMPLES = 32  # Number of calibration samples
OUTPUT_DIR = './entropy_plots'
MODEL_TYPE = 'vit_l'
CHECKPOINT_PATH = './pretrained_checkpoint/sam_hq_vit_l.pth'

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Configuration:")
print(f"  Processor: {PROCESSOR_NAME}")
print(f"  Percent Entropy: {PERCENT_ENTROPY}")
print(f"  Calibration Samples: {NUM_CALIB_SAMPLES}")
print(f"  Output Directory: {OUTPUT_DIR}")

Configuration:
  Processor: POSITIONAL_PRUNE
  Percent Entropy: 0.3
  Calibration Samples: 32
  Output Directory: ./entropy_plots


## 3. Load Configuration and Model

In [3]:
# Load config
config = OmegaConf.load(CONFIG_FILE)
config.quantization.percent_entropy = PERCENT_ENTROPY
config.quantization.high_entropy = True

print("Config loaded:")
print(OmegaConf.to_yaml(config.quantization))

# Initialize model
print(f"\nLoading {MODEL_TYPE} model from {CHECKPOINT_PATH}...")
sam = sam_model_registry[MODEL_TYPE](checkpoint=CHECKPOINT_PATH).to('cuda')
predictor = SamPredictor(sam)
print("Model loaded successfully!")

Config loaded:
quandecoder: false
quanrtn: true
low_high_density: none
quangptq: false
quansmooth: false
quanro: false
rtn_cuda: false
gptq_cuda: false
act_scales_file: ./pretrained_checkpoint/sam_vit_lactivation_scales.pt
act_quant: per_token
weight_quant: per_channel
n_bits: 4
n_bits_mlp: 4
quantize_output: false
up_down_RTN: none
qkT_v: true
channel: false
percent: 100
centerQ: false
percent_entropy: 0.3
high_entropy: true


Loading vit_l model from ./pretrained_checkpoint/sam_hq_vit_l.pth...
<All keys matched successfully>
Model loaded successfully!


## 4. Initialize Engine and Processor

In [4]:
# Initialize engine
print("Initializing engine...")
engine = Engine('entropy_viz', quantize_encoder=True, quantize_decoder=False)

# Get processor
print(f"\nInitializing {PROCESSOR_NAME} processor...")
processor = get_encoder_processor(PROCESSOR_NAME)

# Set up processor
processor.set_dataloaders(engine.dataloaders)
processor.set_accelerator(engine.accelerator)
processor.set_params(config)

print(f"Processor initialized: {processor.strategy_name}")
print(f"  Percent: {processor.percent}")
print(f"  Prune high entropy: {processor.prunehighentropy}")

Initializing engine...
------------------------------ valid --------------------------------
--->>> valid  dataset  0 / 1   DIS5K-VD <<<---
-im- DIS5K-VD ./data/DIS5K/DIS-VD/im :  470
-gt- DIS5K-VD ./data/DIS5K/DIS-VD/gt :  470

Initializing POSITIONAL_PRUNE processor...
------------------------------ valid --------------------------------
--->>> valid  dataset  0 / 1   DIS5K-VD <<<---
-im- DIS5K-VD ./data/DIS5K/DIS-VD/im :  470
-gt- DIS5K-VD ./data/DIS5K/DIS-VD/gt :  470


AttributeError: 'PositionalPruneProcessor' object has no attribute 'set_dataloaders'

## 5. Calibrate Processor to Collect Entropy Statistics

In [5]:
print(f"\n{'='*80}")
print(f"Calibrating {PROCESSOR_NAME} with {NUM_CALIB_SAMPLES} samples...")
print(f"{'='*80}\n")

processor.calibrate(
    predictor=predictor,
    modules=(DecoderAttention, EncoderAttentionTraining, EncoderAttention, EncoderSamAttention),
    num_samples=NUM_CALIB_SAMPLES
)

print("\n" + "="*80)
print("Calibration complete!")
print("="*80)


Calibrating POSITIONAL_PRUNE with 32 samples...

Registering attention hook for blocks.0.attn
Registering attention hook for blocks.1.attn
Registering attention hook for blocks.2.attn
Registering attention hook for blocks.3.attn
Registering attention hook for blocks.4.attn
Registering attention hook for blocks.5.attn
Registering attention hook for blocks.6.attn
Registering attention hook for blocks.7.attn
Registering attention hook for blocks.8.attn
Registering attention hook for blocks.9.attn
Registering attention hook for blocks.10.attn
Registering attention hook for blocks.11.attn
Registering attention hook for blocks.12.attn
Registering attention hook for blocks.13.attn
Registering attention hook for blocks.14.attn
Registering attention hook for blocks.15.attn
Registering attention hook for blocks.16.attn
Registering attention hook for blocks.17.attn
Registering attention hook for blocks.18.attn
Registering attention hook for blocks.19.attn
Registering attention hook for blocks.20

After sample 1: blocks.0.attn.head_0 has 1 entropy values


After sample 2: blocks.0.attn.head_0 has 2 entropy values


After sample 3: blocks.0.attn.head_0 has 3 entropy values


Calculating final entropy variance and mean from accumulated data...


Layer image_encoder.blocks.0.attn: 400 heads with high entropy: tensor([ True,  True,  True,  True,  True,  True,  True,  True, False,  True],
       device='cuda:0')...
Layer image_encoder.blocks.1.attn: 400 heads with high entropy: tensor([ True, False,  True,  True,  True, False,  True, False, False,  True],
       device='cuda:0')...
Layer image_encoder.blocks.2.attn: 400 heads with high entropy: tensor([ True, False,  True, False, False,  True,  True,  True,  True,  True],
       device='cuda:0')...
Layer image_encoder.blocks.3.attn: 400 heads with high entropy: tensor([ True,  True, False,  True,  True, False, False,  True, False, False],
       device='cuda:0')...
Layer image_encoder.blocks.4.attn: 400 heads with high entropy: tensor([False,  True,  True, False, False,  True,  True, False,  True, False],
       device='cuda:0')...
Layer image_encoder.blocks.5.attn: 400 heads with high entropy: tensor([ True, False,  True,  True,  True,  True, False, False, False,  True],
       

## 6. Extract Entropy Statistics

In [6]:
def extract_entropy_statistics(processor):
    """Extract entropy statistics from calibrated processor."""
    if not hasattr(processor, 'entropy_stats'):
        print("Warning: Processor has no entropy_stats attribute")
        return {}

    print(f"\nExtracting entropy statistics from {processor.strategy_name}...")
    print(f"Found {len(processor.entropy_stats)} head entries")

    # Group by layer and head
    layer_head_entropy = {}

    for head_key, entropy_values in processor.entropy_stats.items():
        if len(entropy_values) == 0:
            continue

        # Parse key format: 'blocks.22.attn.head_233'
        parts = head_key.split('.')
        if len(parts) < 4:
            continue

        block_idx = int(parts[1])
        head_idx = int(parts[3].split('_')[1])

        # Calculate mean entropy
        entropy_tensor = torch.tensor(entropy_values) if not isinstance(entropy_values[0], torch.Tensor) else torch.stack(entropy_values)
        mean_entropy = torch.mean(entropy_tensor).item()

        layer_name = f"Block {block_idx}"
        if layer_name not in layer_head_entropy:
            layer_head_entropy[layer_name] = {}

        layer_head_entropy[layer_name][head_idx] = mean_entropy

    print(f"Organized into {len(layer_head_entropy)} layers")
    return layer_head_entropy


def extract_pruning_masks(processor):
    """Extract pruning masks from calibrated processor."""
    if not hasattr(processor, 'final_entropy_stats'):
        print("Warning: Processor has no final_entropy_stats attribute")
        return {}

    print(f"\nExtracting pruning masks...")
    print(f"Found {len(processor.final_entropy_stats)} layers with masks")

    masks = {}
    for layer_name, mask_data in processor.final_entropy_stats.items():
        # Convert layer name format
        parts = layer_name.split('.')
        if 'blocks' in parts:
            block_idx = int(parts[parts.index('blocks') + 1])
            simple_name = f"Block {block_idx}"
            masks[simple_name] = mask_data

    return masks


# Extract statistics
layer_head_entropy = extract_entropy_statistics(processor)
pruning_masks = extract_pruning_masks(processor)

if not layer_head_entropy:
    print("\nError: No entropy statistics found!")
else:
    print(f"\nSuccessfully extracted entropy data for {len(layer_head_entropy)} layers")


Extracting entropy statistics from PositionalHeadPruneProcessor...
Found 8064 head entries
Organized into 24 layers

Extracting pruning masks...
Found 24 layers with masks

Successfully extracted entropy data for 24 layers


## 7. Print Entropy Summary Statistics

In [7]:
print("\n" + "="*80)
print("ENTROPY SUMMARY")
print("="*80)

all_entropies = []
for layer_data in layer_head_entropy.values():
    all_entropies.extend(layer_data.values())

print(f"Total heads/positions tracked: {len(all_entropies)}")
print(f"Mean entropy: {np.mean(all_entropies):.4f}")
print(f"Std entropy: {np.std(all_entropies):.4f}")
print(f"Min entropy: {np.min(all_entropies):.4f}")
print(f"Max entropy: {np.max(all_entropies):.4f}")
print(f"Median entropy: {np.median(all_entropies):.4f}")

if pruning_masks:
    total_pruned = sum(
        mask.sum().item() if isinstance(mask, torch.Tensor) else len(mask)
        for mask in pruning_masks.values()
    )
    print(f"\nTotal pruned elements: {total_pruned}")
    
    # Calculate pruning ratio
    total_elements = len(all_entropies)
    pruning_ratio = total_pruned / total_elements if total_elements > 0 else 0
    print(f"Pruning ratio: {pruning_ratio:.2%}")

print("="*80)


ENTROPY SUMMARY
Total heads/positions tracked: 8064
Mean entropy: 969.2892
Std entropy: 1866.2568
Min entropy: 89.1937
Max entropy: 30765.1094
Median entropy: 850.2011

Total pruned elements: 4032
Pruning ratio: 50.00%
